In [4]:
!pip install PyPDF2
!pip install fitz

ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


In [6]:
!pip install langchain-community

In [7]:
!pip install langchain==0.1.16 langchain-core==0.1.50 langchain-groq==0.1.5


In [8]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


In [9]:
# Load and split the PDF
loader = PyPDFLoader("/kaggle/input/baheshti-zewar-book/BahishtiZewar_text.pdf")
pages = loader.load_and_split()


In [10]:
# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
splits = text_splitter.split_documents(pages)


In [11]:
# Create vector store
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vectorstore = FAISS.from_documents(documents=splits, embedding=embedding_model)


2025-04-30 08:16:03.543034: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746000963.833232      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746000963.920064      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
# Save the FAISS index locally (useful in Kaggle to avoid recomputing)
vectorstore.save_local("faiss_dl_book_index")
# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


In [13]:
# Set up Groq client
groq_api_key = "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"  # Replace with your actual key
llm = ChatGroq(temperature=0, model_name="llama3-70b-8192", groq_api_key=groq_api_key)


In [14]:
# Create prompt template
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# Create RAG chain
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [23]:
question = "List 5 things that break wudu according to Bahishti Zewar."
answer = rag_chain.invoke(question)
print(answer)

Based on the provided context, Bahishti Zewar does not explicitly list 5 things that break wudu. However, it does mention 4 conditions under which ghusl becomes obligatory, which can be inferred to break wudu:

1. Discharge of semen out of excitement.
2. Entry of the glans of the penis into the vagina (or anus).
3. At the end of menstruation.
4. At the end of nifaas (bleeding after childbirth).

Note that wudu and ghusl are related but distinct concepts in Islamic jurisprudence. Wudu is a minor ablution, while ghusl is a major ablution. The conditions mentioned above are related to ghusl, not wudu.


In [29]:
question = "when women can be divorce"
answer = rag_chain.invoke(question)
print(answer)

Based on the provided context, a woman can be divorced in the following situations:

1. If her husband, who is mature, not a lunatic, nor a mad person, divorces her, the divorce will come into effect.
2. If her husband is compelled to divorce her by someone else, and he divorces her due to that compulsion, the divorce will take place.
3. If her husband is under the influence of alcohol or any other intoxicant and divorces her, the divorce will take place.
4. If her husband divorces her in anger, the divorce will take place.
5. If the husband orders someone to divorce his wife, and that person divorces her, the divorce will take place.

Additionally, a woman can be divorced prior to her departure from her parent's home, or after she has gone to her husband's home but they have not met in privacy or seclusion. In such cases, the divorce will take place, and she will not have to complete any 'iddah (waiting period) before remarrying.


In [32]:
import time

In [42]:
import time
from datetime import datetime

def profile_query(question, rag_chain):
    """Most reliable timing method that always works"""
    print(f"\n🔍 Profiling query: '{question[:50]}...'")
    
    # 1. Full pipeline timing
    start_full = time.perf_counter()
    answer = rag_chain.invoke(question)
    full_time = (time.perf_counter() - start_full) * 1000
    
    # 2. Isolated retrieval timing
    start_retrieval = time.perf_counter()
    retrieved_docs = retriever.invoke(question)
    retrieval_time = (time.perf_counter() - start_retrieval) * 1000
    
    # 3. Isolated generation timing (with retrieved docs)
    start_generation = time.perf_counter()
    generation_input = {
        "context": retrieved_docs,
        "question": question
    }
    _ = llm.invoke(prompt.invoke(generation_input))
    generation_time = (time.perf_counter() - start_generation) * 1000
    
    # Calculate overhead (should be minimal)
    overhead = full_time - (retrieval_time + generation_time)
    
    print(f"⏱️  Performance Metrics:")
    print(f"├─ Total Time: {full_time:.2f} ms")
    print(f"├─ Retrieval: {retrieval_time:.2f} ms ({len(retrieved_docs)} docs)")
    print(f"├─ Generation: {generation_time:.2f} ms")
    print(f"└─ System Overhead: {overhead:.2f} ms")
    
    return answer

# Alternative simplified version if you don't have access to internal components
def simple_profile(question, rag_chain):
    """Simpler version when you only have the full chain"""
    start_time = time.perf_counter()
    answer = rag_chain.invoke(question)
    total_time = (time.perf_counter() - start_time) * 1000
    
    print(f"\n⏱️  Query took {total_time:.2f} ms")
    return answer

In [43]:
# Test with your Bahishti Zewar questions
questions = [
    "List 5 things that break wudu according to Bahishti Zewar",
    "What are the rights of husbands mentioned in Bahishti Zewar?",
    "Explain the rules about women traveling without mahram"
]

for question in questions:
    # Use the detailed profiler
    answer = profile_query(question, rag_chain)
    
    # Or use the simple version if preferred
    # answer = simple_profile(question, rag_chain)
    
    print("\nAnswer:", answer[:200] + "...\n")
    print("="*80)


🔍 Profiling query: 'List 5 things that break wudu according to Bahisht...'
⏱️  Performance Metrics:
├─ Total Time: 921.77 ms
├─ Retrieval: 11.80 ms (3 docs)
├─ Generation: 894.04 ms
└─ System Overhead: 15.93 ms

Answer: Based on the provided context, Bahishti Zewar does not explicitly list 5 things that break wudu. However, it does mention 4 conditions under which ghusl becomes obligatory, which can be inferred to br...


🔍 Profiling query: 'What are the rights of husbands mentioned in Bahis...'
⏱️  Performance Metrics:
├─ Total Time: 508.37 ms
├─ Retrieval: 12.13 ms (3 docs)
├─ Generation: 507.76 ms
└─ System Overhead: -11.52 ms

Answer: Based on the provided context, there is no mention of the rights of husbands in Bahishti Zewar. The text only mentions the rights of Muslims in general (on page 480) and does not specifically mention ...


🔍 Profiling query: 'Explain the rules about women traveling without ma...'
⏱️  Performance Metrics:
├─ Total Time: 551.43 ms
├─ Retrieval: 11.29 m

In [15]:
# After creating your FAISS index
print(f"Total vectors in index: {vectorstore.index.ntotal}")
print(f"Vector dimensions: {vectorstore.index.d}")

Total vectors in index: 2957
Vector dimensions: 384


In [16]:
print(f"Number of document chunks: {len(splits)}")

Number of document chunks: 2957


In [17]:
test_embedding = embedding_model.embed_query("test")
print(f"Embedding dimensions: {len(test_embedding)}")

Embedding dimensions: 384
